[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamaravdd/iese-dsmba/blob/main/notebooks/500-Reddit-Scraping-Merging.ipynb)

# Web Scraping & Data Merging

Every day, millions of people share opinions, ask questions, and discuss products on the open web — on Reddit, review sites, forums, and news pages. That data is publicly visible and enormously valuable for business intelligence.

In this notebook we will:

1. **Scrape** static HTML with `requests` + `BeautifulSoup`
2. **Query** Reddit's API using `PRAW` to find posts mentioning specific car brands
3. **Merge** Reddit buzz data with our `cars.csv` dataset and OEM margin data using `pd.merge`
4. **Analyse** whether heavily-discussed brands also command higher prices or margins

**Required packages** — run once:
```bash
pip install requests beautifulsoup4 praw pandas matplotlib seaborn
```

In [8]:
# run this once, then comment it out! 
pip3 install requests beautifulsoup4 praw pandas matplotlib seaborn

SyntaxError: invalid syntax (3918792087.py, line 2)

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

pd.set_option('display.max_colwidth', 80)

---
## Part I: Web Scraping with requests + BeautifulSoup

### How a web page works

When you visit a URL in your browser, your browser sends an **HTTP GET request** to a server. The server responds with **HTML** — a text document that describes the page structure. Your browser then renders that HTML into the visual page you see.

`requests.get(url)` does exactly the same thing — minus the rendering. You get the raw HTML back as a string, which you can then parse with **BeautifulSoup**.

```
Your Python script  →  HTTP GET request  →  Web server
                    ←  HTML response     ←
BeautifulSoup parses the HTML
You extract the data you need
```

### A minimal HTML primer

HTML is a tree of **tags**. Every piece of content lives inside one or more tags:

```html
<div class="product-card">
    <h2>Alfa Romeo 4C</h2>
    <p class="price">€75,360</p>
    <a href="/cars/alfa-romeo-4c">View details</a>
</div>
```

BeautifulSoup lets you navigate this tree:
- `soup.find('h2')` → first `<h2>` tag
- `soup.find_all('p', class_='price')` → all `<p>` tags with class `price`
- `tag.text` → the visible text inside the tag
- `tag['href']` → the value of an attribute

In [2]:
# Let's fetch a real page: the Hacker News front page
# Hacker News is a tech-oriented news aggregator — it's been around since 2007
# and its clean HTML makes it a classic scraping exercise

url = 'https://news.ycombinator.com/'
headers = {'User-Agent': 'Mozilla/5.0 (educational scraping exercise)'}  # polite to identify yourself

response = requests.get(url, headers=headers, timeout=10)
print(f"Status: {response.status_code}")   # 200 = OK
print(f"Content length: {len(response.text):,} characters")
print("\nFirst 500 characters of HTML:")
print(response.text[:500])

Status: 200
Content length: 34,439 characters

First 500 characters of HTML:
<html lang="en" op="news"><head><meta name="referrer" content="origin"><meta name="viewport" content="width=device-width, initial-scale=1.0"><link rel="stylesheet" type="text/css" href="news.css?CeGOgNoXhCGkeRXBPrcN"><link rel="icon" href="y18.svg"><link rel="alternate" type="application/rss+xml" title="RSS" href="rss"><title>Hacker News</title></head><body><center><table id="hnmain" border="0" cellpadding="0" cellspacing="0" width="85%" bgcolor="#f6f6ef"><tr><td bgcolor="#ff6600"><table border=


> ✅ **Sanity check:** Status code should be `200`. If you get `403` or `429`, the server blocked you — try adding a more realistic User-Agent string. A content length of ~50,000 characters is typical for HN.

> 🧠 **Pause and Think:** Before looking at the HTML — what tags do you expect story titles to be wrapped in? A heading tag like `<h2>`? Something with a class? Make a prediction before reading the code.

In [3]:
# Parse with BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# HN story titles live in <span class="titleline"> → <a> tags
stories = soup.find_all('span', class_='titleline')

print(f"Found {len(stories)} stories\n")

# Extract title and link for each story
hn_data = []
for story in stories:
    a_tag = story.find('a')
    if a_tag:
        hn_data.append({
            'title': a_tag.text,
            'url': a_tag.get('href', '')
        })

hn_df = pd.DataFrame(hn_data)
hn_df.head(10)

Found 30 stories



,title,url
0,Claude Fable 5,https://www.anthropic.com/news/claude-fable-5-mythos-5
1,Apple decided not to roll out Siri in EU after denied request for exemption,https://www.reuters.com/business/apple-failed-make-its-ai-tool-comply-eu-reg...
2,System Card: Claude Fable 5 and Claude Mythos 5 [pdf],https://www-cdn.anthropic.com/d00db56fa754a1b115b6dd7cb2e3c342ee809620.pdf
3,Where is the AI jobs crisis?,https://www.apollo.com/wealth/the-daily-spark/where-is-the-ai-jobs-crisis
4,Making Graphics Like it's 1993,https://staniks.github.io/articles/catlantean-3d-blog-1/
5,A giant star may have destroyed itself in one of the rarest explosions,https://phys.org/news/2026-05-giant-star-destroyed-universe-rarest.html
6,Microsoft's open source tools were hacked to steal passwords of AI developers,https://techcrunch.com/2026/06/08/microsofts-open-source-tools-were-hacked-t...
7,Launch HN: Transload (YC P26) – Measuring freight items with CCTV,item?id=48463273
8,Biff.core: system composition for Clojure web apps,https://biffweb.com/p/core/
9,The LD_DEBUG environment variable (2012),https://bnikolic.co.uk/blog/linux-ld-debug.html


>✏️ **You try it**
>
> HN also shows each story's score (upvotes). Can you find and extract those too?
>
> Hint: Inspect the HTML source (right-click → View Page Source in your browser) and look for `<span class="score">`.
> Try to add a `score` column to `hn_df`.

In [4]:
# Your code here
# Hint: soup.find_all('span', class_='score') gives you the score tags
# Each tag's .text looks like '312 points'


---
## Part II: Reddit API with PRAW

### Why use an API instead of scraping HTML?

Reddit's website is heavily JavaScript-rendered — scraping its HTML is messy. But Reddit provides an **official API**: instead of parsing HTML, you get back clean JSON with exactly the fields you need.

**PRAW** (Python Reddit API Wrapper) is the standard library for Reddit's API. It handles authentication, rate limiting, and pagination for you.

### Setup

1. Go to [https://www.reddit.com/prefs/apps](https://www.reddit.com/prefs/apps)
2. Click **"create another app"** → choose **"script"**
3. Name: `iese-scraping-exercise`, redirect URI: `http://localhost:8080`
4. Copy the `client_id` (below the app name) and `client_secret`

Then fill in the credentials below.

> **No credentials?** No problem — skip to the backup cell below to load a pre-scraped dataset and continue from Part III.

In [5]:
import praw

# Fill in your Reddit API credentials here
reddit = praw.Reddit(
    client_id     = 'YOUR_CLIENT_ID',
    client_secret = 'YOUR_CLIENT_SECRET',
    user_agent    = 'iese-dsmba-exercise:v1.0 (by /u/YOUR_REDDIT_USERNAME)',
    # If you want to post/vote, add username + password. For reading, these aren't needed.
)

# Test: get the top 3 posts in r/datascience
subreddit = reddit.subreddit('datascience')
for post in subreddit.hot(limit=3):
    print(f"[{post.score:>6}] {post.title[:80]}")

ModuleNotFoundError: No module named 'praw'

> ✅ **Sanity check:** You should see 3 post titles printed with their scores. If you get an `OAuthException` or `prawcore.exceptions.ResponseException`, double-check your `client_id` and `client_secret` — they're easy to mix up. Use the backup cell if you can't resolve it.

### The anatomy of a Reddit post (PRAW submission)

A PRAW `submission` object has many attributes:

| Attribute | What it contains |
|---|---|
| `submission.title` | Post title |
| `submission.selftext` | Body text (empty for link posts) |
| `submission.score` | Net upvotes |
| `submission.num_comments` | Comment count |
| `submission.created_utc` | Unix timestamp |
| `submission.url` | URL (link posts) or Reddit permalink |
| `submission.subreddit.display_name` | Which subreddit |
| `submission.author.name` | Username (may be deleted) |

> 🧠 **Pause and Think:** Before we scrape: which of the 10 brands do you expect to generate the most Reddit discussion? Rank your top 3. Then check after — were you right? What might explain the gaps?

In [ ]:
# The brands we want to track — same ones in our cars.csv dataset
car_brands = [
    'BMW', 'Mercedes', 'Audi', 'Volkswagen', 'Toyota',
    'Honda', 'Porsche', 'Volvo', 'Ford', 'Lamborghini'
]

def scrape_brand_mentions(reddit_client, brand, subreddits=['cars', 'whatcarshouldibuy', 'askcarsales'], limit=50):
    """
    Search Reddit for posts mentioning a car brand.
    Returns a list of dicts with post metadata.
    """
    results = []
    for sub_name in subreddits:
        sub = reddit_client.subreddit(sub_name)
        try:
            for post in sub.search(brand, limit=limit, sort='relevance', time_filter='year'):
                results.append({
                    'brand':        brand,
                    'subreddit':    sub_name,
                    'title':        post.title,
                    'score':        post.score,
                    'num_comments': post.num_comments,
                    'created_utc':  post.created_utc,
                })
        except Exception as e:
            print(f"  Error scraping r/{sub_name} for {brand}: {e}")
        time.sleep(0.5)  # be polite — don't hammer the API
    return results


# Scrape all brands (this takes ~1-2 minutes)
all_posts = []
for brand in car_brands:
    print(f"Scraping: {brand} ...", end=' ')
    posts = scrape_brand_mentions(reddit, brand)
    all_posts.extend(posts)
    print(f"{len(posts)} posts")

reddit_df = pd.DataFrame(all_posts)
print(f"\nTotal posts collected: {len(reddit_df)}")
reddit_df.head()

### 🔁 Backup: Load pre-scraped data

If you don't have Reddit API credentials yet, run the cell below instead of the scraping cell above.
It loads an identical dataset collected in advance — same columns, same structure, so all the code below works without changes.

In [ ]:
# ── BACKUP: run this cell instead of the scraping cell above ──────────────────
# Pre-scraped Reddit posts for 10 car brands (r/cars, r/whatcarshouldibuy, r/askcarsales)

BACKUP_URL = 'https://raw.githubusercontent.com/tamaravdd/iese-dsmba/main/resources/tabular/reddit_cars_prescrape.csv'

reddit_df = pd.read_csv(BACKUP_URL)

# The live scrape produces a created_utc column; replicate that here
reddit_df['date'] = pd.to_datetime(reddit_df['created_utc'], unit='s')

print(f"Loaded {len(reddit_df)} posts from backup dataset")
reddit_df.head()

In [ ]:
# Convert Unix timestamp to a proper date
reddit_df['date'] = pd.to_datetime(reddit_df['created_utc'], unit='s')

# Basic exploration
print("Posts per brand:")
print(reddit_df['brand'].value_counts())
print("\nPosts per subreddit:")
print(reddit_df['subreddit'].value_counts())

> ✅ **Sanity check:** You should have roughly 5–8 posts per brand per subreddit (given the `limit=50` cap). If one brand has 0 posts, the search term may not match Reddit's index — try a variant (e.g. `'VW'` for Volkswagen). Also check: `created_utc` values should be Unix timestamps (large integers), not already-formatted dates.

In [ ]:
# Aggregate: compute a 'buzz score' per brand
# We'll use: total posts + total comments + total score (upvotes)

buzz = reddit_df.groupby('brand').agg(
    post_count    = ('title',        'count'),
    total_score   = ('score',        'sum'),
    total_comments= ('num_comments', 'sum'),
    avg_score     = ('score',        'mean'),
).reset_index()

# Normalise to a composite buzz index (0-100)
buzz['buzz_index'] = (
    buzz['post_count'] / buzz['post_count'].max() * 0.3 +
    buzz['total_score'] / buzz['total_score'].max() * 0.4 +
    buzz['total_comments'] / buzz['total_comments'].max() * 0.3
) * 100

buzz = buzz.sort_values('buzz_index', ascending=False)
buzz

>✏️ **You try it**
>
> 1. Which brand has the highest average score per post? What might that tell you?
> 2. Create a horizontal bar chart of `buzz_index` per brand.
> 3. Are there posts in the dataset where our target brand isn't actually mentioned in the title? Can you filter those out?
>
> *Hint for 3: use `str.contains(brand, case=False)` on the title column.*

In [ ]:
# Your code here


---
## Part III: Data Merging

### Why merge?

Our Reddit buzz data tells us **how much people talk** about each brand. Our cars dataset tells us **what those cars cost and how powerful they are**. Our OEM margin data tells us **how profitable each manufacturer is**. Each dataset alone is useful — together they become far more powerful.

This pattern — enriching one dataset with information from another — is one of the most common operations in real-world data science.

### How pd.merge works

```python
result = left_df.merge(right_df, on='key_column', how='left')
```

- `on=` specifies which column(s) to match on
- `how=` controls what happens when there's no match:
  - `'left'` — keep all rows from `left_df`, fill `NaN` if no match in right
  - `'inner'` — keep only rows that match in both
  - `'outer'` — keep all rows from both, fill `NaN` where no match

In [ ]:
# Load the cars dataset
cars_df = pd.read_csv('https://raw.githubusercontent.com/ciri/iese-dsfb/main/resources/tabular/cars.csv')

print(f"Cars dataset: {cars_df.shape}")
cars_df.head(3)

In [ ]:
# To merge, we need a common key. Our buzz data has 'brand' (e.g. 'BMW').
# The cars dataset also has 'brand'. Let's check they match.

print("Brands in buzz data:", sorted(buzz['brand'].unique()))
print("\nSample brands in cars data:", sorted(cars_df['brand'].unique())[:15])

In [ ]:
# Step 1: Compute average price per brand from the cars dataset
brand_stats = cars_df.groupby('brand').agg(
    avg_price       = ('price',              'mean'),
    avg_hp          = ('total_max_power_hp', 'mean'),
    model_count     = ('model_name',         'count'),
).reset_index()

brand_stats.head(5)

> 🧠 **Pause and Think:** We're about to merge `buzz` (10 brands from Reddit) with `brand_stats` (591 models from cars.csv). Should we use `how='left'`, `'inner'`, or `'outer'`? What happens to a brand like 'Lamborghini' if it's in buzz but its model name in cars.csv uses a different spelling?

In [ ]:
# Step 2: Merge buzz with brand_stats
# We use how='left' to keep all buzz brands even if not in cars.csv

merged = buzz.merge(brand_stats, on='brand', how='left')

print(f"Rows matched: {merged['avg_price'].notna().sum()} / {len(merged)}")
merged

> ✅ **Sanity check:** The print statement tells you how many rows matched. You should expect 8–10 out of 10 to match. Any `NaN` in `avg_price` means that brand name in Reddit didn't match the cars.csv spelling exactly — spot-check with `merged[merged['avg_price'].isna()]`.

In [ ]:
# Step 3: Load OEM gross margin data and add to our merged table
# This data lives at the 'group' level (e.g. Volkswagen Group includes Audi, Porsche, VW)

margins_df = pd.read_csv('https://raw.githubusercontent.com/ciri/iese-dsfb/main/resources/tabular/car_margins.csv')
print("Margin data:")
print(margins_df)

# We need to link brands to groups
# Let's get that from cars_df
brand_to_group = cars_df[['brand','group']].drop_duplicates()
brand_to_group.head()

In [ ]:
# Chain two merges: buzz → brand_stats → group → margins
full = (
    buzz
    .merge(brand_stats,    on='brand', how='left')
    .merge(brand_to_group, on='brand', how='left')
    .merge(margins_df,     on='group', how='left')
)

print(f"Final dataset shape: {full.shape}")
full[['brand', 'buzz_index', 'avg_price', 'avg_hp', 'group', 'oem_gross_margin']].sort_values('buzz_index', ascending=False)

> ✅ **Sanity check:** Run `full.isna().sum()` — some brands may have `NaN` for `oem_gross_margin` if they don't belong to a group in the margins table. That's expected. Also check `full.shape`: you should still have exactly one row per brand (10 rows), not duplicates from the join.

---
## Part IV: Analysis — Reddit Buzz vs Business Metrics

Now we have a single table connecting Reddit buzz to real business data. Let's explore.

**Central question:** Does more Reddit discussion correlate with higher prices, more power, or higher margins?

In [ ]:
# Scatter: buzz index vs average price
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (col, label) in zip(axes, [('avg_price', 'Average Price (€)'), ('avg_hp', 'Average Horsepower')]):
    subset = full.dropna(subset=['buzz_index', col])
    ax.scatter(subset['buzz_index'], subset[col], s=80, color='steelblue', alpha=0.7)
    for _, row in subset.iterrows():
        ax.annotate(row['brand'], (row['buzz_index'], row[col]), fontsize=8, ha='center', va='bottom')
    ax.set_xlabel('Buzz Index (Reddit)', fontsize=12)
    ax.set_ylabel(label, fontsize=12)
    ax.set_title(f'Buzz vs {label}', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation table
full[['buzz_index', 'avg_price', 'avg_hp', 'oem_gross_margin']].corr().round(3)

>✏️ **You try it**
>
> 1. Which brand has the highest buzz-to-price ratio? (A brand that generates a lot of discussion relative to its price)
> 2. Which brand has the lowest?
> 3. Compute `estimated_profit_per_car = avg_price × oem_gross_margin`. Which brand generates the most profit per car AND has high buzz? Is there a sweet spot?
> 4. Use your favourite LLM to help you plot a bubble chart where: x = buzz_index, y = avg_price, bubble size = model_count, colour = oem_gross_margin.
>
> *Hint: copy the `full` dataframe description and the columns to your LLM and ask it to write the matplotlib/seaborn code.*

In [ ]:
# Your code here


---
## Bonus: Scraping Without an API

Not every site has a convenient API. When you're dealing with static HTML, `requests` + `BeautifulSoup` is the standard toolchain.

The general workflow:
1. Open the page in your browser and use **Inspect Element** (F12) to find the HTML structure
2. Identify the tags and class names that wrap the data you want
3. Write a `find_all()` to capture all instances
4. Extract `.text` or tag attributes
5. Handle pagination (look for a "next page" link or URL pattern like `?page=2`)

**Rate-limit yourself:** always add a `time.sleep(1)` between requests. You're a guest on their server.

In [ ]:
# Example: scraping multiple pages of Hacker News
# HN paginates with ?p=2, ?p=3, ...

all_titles = []

for page in range(1, 4):  # pages 1-3
    url = f'https://news.ycombinator.com/?p={page}'
    resp = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    for span in soup.find_all('span', class_='titleline'):
        a = span.find('a')
        if a:
            all_titles.append({'page': page, 'title': a.text, 'url': a.get('href', '')})
    
    print(f"Page {page}: {len(all_titles)} total titles")
    time.sleep(1.0)  # be polite

hn_pages_df = pd.DataFrame(all_titles)
print(f"\nTotal: {len(hn_pages_df)} titles across 3 pages")
hn_pages_df.tail(5)

## Final Note

Web scraping and data merging are the **data collection backbone** of business intelligence.

- When an API exists, use it — it's cleaner, faster, and more respectful of the site.
- When you're scraping HTML, inspect the page structure carefully and be polite with rate limits.
- Merging is almost always necessary in practice: no single dataset contains everything you need.
- Once you have a merged dataset, the real work begins — visualising, modelling, and communicating insights.

The Reddit → cars → margins pipeline we built today is a miniature version of what data teams at hedge funds, brand agencies, and automotive companies do every day.